# Data Science 1 · Linear and logistic regression

**Lesson:** Data Science 1, Day 1 · **Time:** about 15 minutes · **ISLP:** 3.1–3.3, 4.1–4.3 · **Data:** CADets release 1

**What you will do**
- Predict length of stay: first with the mean, then with a line by age decade
- Split one patient's stay into the mean, what the line explains and the error; then do it for 50 patients and get R²
- Go from the odds ratio of a 2×2 table to a logistic regression, and calculate a patient's risk of death

We follow one patient from the slides all the way: **P00503**, age 84, admitted with an infection, who stayed 11.0 days.

Run each cell in order with **Shift + Enter**. Cells marked **Your turn** have a blank (`____`) to fill in.

**The packages in this notebook**

| Package | Short name | What we use it for |
|---|---|---|
| pandas | `pd` | Tables: read the data, pick columns, group and count |
| NumPy | `np` | Math on numbers: `np.exp` (e to the power) and `np.log` |
| matplotlib | `plt` | Plots |
| statsmodels | `smf` | Regression models, written as a formula such as `"los_days ~ age_decade"` |

In [ ]:
# Setup: load the CADets course dataset. Run this cell first.
# RELEASE: 1 from Day 1, 2 from Day 2, 3 from Day 5. Use the release participants have on your lesson day.
RELEASE = 1

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {1: "cadets_admissions_release1_core.csv",
         2: "cadets_admissions_release2_intervention.csv",
         3: "cadets_admissions_release3_full.csv"}
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/"

def load_cadets(release=RELEASE):
    name = FILES[release]
    if os.path.exists(name):  # a copy uploaded to this session
        return pd.read_csv(name)
    try:  # the course repository, from the release day on
        return pd.read_csv(DATA_URL + name)
    except Exception:
        pass
    try:  # before the release day: upload the file from the course website's dataset page
        from google.colab import files
        print(f"Release {release} is not published yet. Upload {name} from the course website's dataset page.")
        files.upload()
        return pd.read_csv(name)
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the same folder as this notebook.")

df = load_cadets()
print(f"Release {RELEASE}: {df.shape[0]:,} admissions, {df.shape[1]} columns")

### Plot settings

Run this cell; you do not need to read it. It sets the lesson colours, adds the age decade (8 = age 80 to 89), picks the 50 patients shown on the slides, and defines `draw_patients`, which draws them.

In [ ]:
import statsmodels.formula.api as smf

TEAL, VIOLET, CORAL, NAVY = "#007A8C", "#5B2EFF", "#FF4D6D", "#1B1446"
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False})

df["age_decade"] = df["age"] // 10           # // divides and drops the remainder: 84 // 10 = 8
ids_50 = [
    'P01680', 'P01868', 'P01617', 'P00497', 'P01515', 'P01679', 'P00488', 'P01257', 'P00486', 'P01801',
    'P00473', 'P00316', 'P01995', 'P00045', 'P00075', 'P00614', 'P00901', 'P00851', 'P00224', 'P01804',
    'P01148', 'P01020', 'P01045', 'P01528', 'P01158', 'P01042', 'P00344', 'P00019', 'P00984', 'P01133',
    'P01877', 'P01908', 'P01595', 'P00256', 'P00990', 'P00541', 'P01202', 'P01284', 'P00177', 'P01968',
    'P01829', 'P01155', 'P00670', 'P01521', 'P00872', 'P00395', 'P00562', 'P00340', 'P00915', 'P00503',
]
sample_50 = df[df["patient_id"].isin(ids_50)].copy()
sample_50["x"] = sample_50["age_decade"] + np.random.default_rng(4).uniform(-0.27, 0.27, 50)
sample_50.loc[sample_50["patient_id"] == "P00503", "x"] = 8
patient_503 = df[df["patient_id"] == "P00503"]

def draw_patients(ax):
    ax.scatter(sample_50["x"], sample_50["los_days"], s=40, color=TEAL, alpha=0.6)
    ax.scatter(8, 11.0, s=110, color=NAVY, zorder=5)
    ax.annotate("P00503", (8, 11.0), xytext=(-70, 6), textcoords="offset points", color=NAVY, fontweight="bold")
    ax.set_xlim(0, 10.5); ax.set_ylim(0, 20)
    ax.set_xlabel("Age decade (8 = age 80-89)"); ax.set_ylabel("Length of stay (days)")

## Step 1 · Without a model: the mean

`df["los_days"]` picks one column from the table, and `.mean()` (pandas) gives its average. Without any model, the mean is the best single guess for every patient.

In [ ]:
mean_los = df["los_days"].mean()
print("Mean length of stay:", round(mean_los, 2), "days")
patient_503[["patient_id", "age", "age_decade", "heart_failure", "admit_diagnosis_group", "systolic_bp", "los_days"]]

In [ ]:
fig, ax = plt.subplots()
draw_patients(ax)
ax.axhline(mean_los, color=CORAL, lw=3)          # axhline (matplotlib): a horizontal line across the plot
ax.text(0.2, mean_los + 0.4, f"Mean = {mean_los:.1f} days", color=CORAL, fontweight="bold")
ax.set_title("50 patients and the mean: the same guess for everyone")
plt.show()

Is there a pattern by age? `.groupby("age_decade")` (pandas) splits the table into one group per decade; `.mean()` then averages each group. The twenties hold a single patient, so we start from the thirties.

In [ ]:
by_decade = df[df["age_decade"] >= 3].groupby("age_decade")["los_days"].mean()
fig, ax = plt.subplots()
ax.bar(by_decade.index, by_decade.values, color=VIOLET)
ax.axhline(mean_los, color=NAVY, ls="--", label="Overall mean")
ax.set_xlabel("Age decade"); ax.set_ylabel("Mean length of stay (days)")
ax.set_title("The mean stay rises with age"); ax.legend()
plt.show()

## Step 2 · A line: intercept and slope

`smf.ols` (statsmodels) fits a linear regression. OLS stands for *ordinary least squares*: of all straight lines, the one closest to all the patients. The formula `"los_days ~ age_decade"` reads "length of stay predicted by age decade".

- `.fit()` runs the calculation on the data.
- `.params` holds the two numbers that describe the line: the intercept and the slope.

In [ ]:
model_age = smf.ols("los_days ~ age_decade", data=df).fit()
b0, b1 = model_age.params["Intercept"], model_age.params["age_decade"]
print(f"β₀ (intercept) = {b0:.2f} days      β₁ (slope) = {b1:.2f} days per decade")

In [ ]:
decades = np.linspace(0, 10, 50)                # 50 evenly spaced numbers from 0 to 10
fig, ax = plt.subplots()
draw_patients(ax)
ax.axhline(mean_los, color=NAVY, ls="--", lw=1.5, label="Mean")
ax.plot(decades, b0 + b1 * decades, color=CORAL, lw=3, label="Regression line")
ax.scatter(0, b0, s=120, color=VIOLET, zorder=5)
ax.annotate(f"β₀ = intercept = {b0:.2f}", (0, b0), xytext=(10, 10), textcoords="offset points", color=VIOLET, fontweight="bold")
ax.plot([5, 8, 8], [b0 + b1 * 5, b0 + b1 * 5, b0 + b1 * 8], color=VIOLET, lw=2.5)
ax.text(6.5, b0 + b1 * 5 - 1.3, "+3 decades", color=VIOLET, fontweight="bold", ha="center")
ax.text(8.15, b0 + b1 * 6.5, f"+3 × β₁ = +{3 * b1:.1f} days", color=VIOLET, fontweight="bold")
ax.set_title("LOS = β₀ (intercept) + β₁ (slope) × age decade"); ax.legend(loc="upper left")
plt.show()

LOS = β₀ + β₁ × age decade, so here LOS = 1.08 + 0.70 × age decade.

- **β₀, the intercept (1.08):** the predicted stay at age 0, where the line meets the vertical axis. Nobody is that young; it anchors the line.
- **β₁, the slope (0.70):** the coefficient of age. Each additional decade of age adds 0.70 days.

### Your turn 1

`.predict()` (statsmodels) gives the model's prediction for patients we describe in a small table; `pd.DataFrame` (pandas) builds that table. Predict the stay of patient P00503, age 84: replace the blank with the age decade.

In [ ]:
patient = pd.DataFrame({"age_decade": [____]})
line_503 = model_age.predict(patient)[0]
print("Predicted stay:", round(line_503, 2), "days.   Actual stay: 11.0 days")

## Step 3 · How much does the line explain? R²

First one patient. P00503's stay = the mean + what the line adds + the error.

In [ ]:
line_503 = model_age.predict(patient_503).iloc[0]
print(f"Mean {mean_los:.2f}  ->  line {line_503:.2f}  ->  actual 11.00")
print(f"Explained by the line = line - mean   = {line_503 - mean_los:.2f}")
print(f"Error                 = actual - line = {11.0 - line_503:.2f}")

fig, ax = plt.subplots()
draw_patients(ax)
ax.axhline(mean_los, color=NAVY, ls="--", lw=1.5)
ax.plot(decades, b0 + b1 * decades, color=CORAL, lw=3)
ax.vlines(8, mean_los, line_503, color=VIOLET, lw=8, label=f"Explained by the line: {line_503 - mean_los:.1f}")
ax.vlines(8, line_503, 11.0, color=CORAL, lw=4, label=f"Error: {11.0 - line_503:.1f}")
ax.set_title("P00503: 11.0 = 5.7 + 1.0 + 4.3"); ax.legend(loc="upper left")
plt.show()

Now the same split for all 50 patients. Square each part, so that positive and negative distances do not cancel, and add them up.

In [ ]:
line_50 = model_age.predict(sample_50)
explained = ((line_50 - mean_los) ** 2).sum()
error = ((sample_50["los_days"] - line_50) ** 2).sum()
total = ((sample_50["los_days"] - mean_los) ** 2).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [2, 1]})
draw_patients(ax1)
ax1.axhline(mean_los, color=NAVY, ls="--", lw=1.5)
ax1.plot(decades, b0 + b1 * decades, color=CORAL, lw=3)
ax1.vlines(sample_50["x"], mean_los, line_50, color=VIOLET, lw=4)
ax1.vlines(sample_50["x"], line_50, sample_50["los_days"], color=CORAL, lw=1.5, alpha=0.7)
ax1.set_title("Violet: explained by the line.  Red: error")
ax2.bar(["Explained", "Error", "Total"], [explained, error, total], color=[VIOLET, CORAL, NAVY])
for i, v in enumerate([explained, error, total]):
    ax2.text(i, v + 8, f"{v:.0f}", ha="center", fontweight="bold")
ax2.set_title("Sums of squares, 50 patients")
plt.tight_layout(); plt.show()

print(f"Explained {explained:.0f} + error {error:.0f} = total {total:.0f}")
print("R² = explained / total =", round(explained / total, 3))
print("R² for all 2,000 patients (model_age.rsquared):", round(model_age.rsquared, 3))

Age decade explains 8% of the variation in length of stay: the violet part is small next to the red.

## Step 4 · Many predictors: binary, categorical and continuous

The same formula, with more terms joined by `+`:
- heart failure is **binary** (0/1);
- admission diagnosis is **categorical**: `C(...)` in the formula turns it into one 0/1 column per group, each compared with the reference group, cardiac;
- systolic blood pressure is **continuous**.

In [ ]:
model_los = smf.ols("los_days ~ age_decade + heart_failure + C(admit_diagnosis_group) + systolic_bp", data=df).fit()
comparison = pd.DataFrame({"Age only": model_age.params, "Four predictors": model_los.params})
print(comparison.round(4).fillna("-"))
print("R²:", round(model_age.rsquared, 3), "->", round(model_los.rsquared, 3))

Each coefficient is the change in days when its predictor goes up by 1, with the other predictors held fixed. The age coefficient changes from 0.70 to 0.61 because it is now adjusted for the others. The coefficient is the same for every patient; the predictor is the patient's own value.

In [ ]:
labels = {"heart_failure": "Heart failure: yes vs no", "C(admit_diagnosis_group)[T.infection]": "Infection vs cardiac",
          "C(admit_diagnosis_group)[T.respiratory]": "Respiratory vs cardiac", "C(admit_diagnosis_group)[T.other]": "Other vs cardiac",
          "C(admit_diagnosis_group)[T.gi]": "GI vs cardiac", "age_decade": "Age: per decade", "systolic_bp": "Systolic BP: per 1 mmHg"}
coefs = model_los.params[list(labels)].rename(labels).sort_values()
fig, ax = plt.subplots()
ax.barh(coefs.index, coefs.values, color=[CORAL if v < 0 else VIOLET for v in coefs.values])
ax.axvline(0, color=NAVY, lw=1)
for i, v in enumerate(coefs.values):
    ax.text(max(v, 0) + 0.02, i, f"{v:+.3g}", va="center")
ax.set_xlabel("Change in predicted stay (days) for one unit of the predictor")
ax.set_title("Coefficients of the four-predictor model")
plt.show()

In [ ]:
steps = {"Mean": mean_los, "Age only": model_age.predict(patient_503).iloc[0],
         "Four predictors": model_los.predict(patient_503).iloc[0], "Actual": 11.0}
fig, ax = plt.subplots()
ax.bar(list(steps), list(steps.values()), color=[NAVY, CORAL, VIOLET, TEAL])
for i, v in enumerate(steps.values()):
    ax.text(i, v + 0.2, f"{v:.1f}", ha="center", fontweight="bold")
ax.set_ylabel("Days"); ax.set_title("P00503: each model gets closer to the actual stay")
plt.show()

## Step 5 · A yes/no outcome: from the odds ratio to logistic regression

`pd.crosstab` (pandas) counts patients by two columns at once: here, heart failure and in-hospital death. This is the 2×2 table from the epidemiology refresher, with the usual letters a, b, c, d.

In [ ]:
table = pd.crosstab(df["heart_failure"], df["in_hospital_death"])
print(table)
a, b = table.loc[1, 1], table.loc[1, 0]   # heart failure: died, survived
c, d = table.loc[0, 1], table.loc[0, 0]   # no heart failure: died, survived
odds_ratio = (a * d) / (b * c)
print("OR = ad / bc =", round(odds_ratio, 2), "   log(OR) =", round(np.log(odds_ratio), 3))   # np.log (NumPy): natural log

rates = df.groupby("heart_failure")["in_hospital_death"].mean() * 100
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["No heart failure", "Heart failure"], rates.values, color=[TEAL, CORAL])
for i, v in enumerate(rates.values):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("Died in hospital (%)"); ax.set_title("Death rate by heart failure")
plt.show()

`smf.logit` (statsmodels) fits a logistic regression: the same formula style, for a 0/1 outcome. `fit(disp=0)` just hides the fitting messages. `np.exp` (NumPy) is e to the power, the reverse of `np.log`: it turns the coefficient back into an OR.

In [ ]:
model_hf = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)
print(model_hf.params.round(3))
print("e to the power β =", round(np.exp(model_hf.params["heart_failure"]), 2), "= the OR of the table")

Risk in three steps: add up z (the log-odds) from the coefficients; odds = e<sup>z</sup>; risk = odds ÷ (1 + odds). Together, these steps are the logistic function.

In [ ]:
g0, g1 = model_hf.params["Intercept"], model_hf.params["heart_failure"]
for hf in [0, 1]:
    z = g0 + g1 * hf
    odds = np.exp(z)
    print(f"Heart failure = {hf}:  z = {z:.2f}   odds = {odds:.3f}   risk = {odds / (1 + odds):.3f}")

z_grid = np.linspace(-6, 6, 200)
fig, ax = plt.subplots()
ax.plot(z_grid, 1 / (1 + np.exp(-z_grid)), color=VIOLET, lw=3)
for hf, colour, name in [(0, TEAL, "No heart failure"), (1, CORAL, "Heart failure")]:
    z = g0 + g1 * hf
    ax.scatter(z, 1 / (1 + np.exp(-z)), s=120, color=colour, zorder=5, label=f"{name}: z = {z:.2f}, risk = {1 / (1 + np.exp(-z)):.1%}")
ax.set_xlabel("z (log-odds)"); ax.set_ylabel("Risk of death"); ax.set_ylim(0, 1)
ax.set_title("The logistic function turns any z into a risk between 0 and 1"); ax.legend(loc="upper left")
plt.show()

The model's risks are exactly the death rates in the two rows of the table.

## Step 6 · Age, then many predictors

A continuous predictor: the coefficient is per decade of age. The dots are the death rate in each decade (bigger dot = more patients); the line is the model.

In [ ]:
model_dec = smf.logit("in_hospital_death ~ age_decade", data=df).fit(disp=0)
print(model_dec.params.round(3))
print("OR per decade:", round(np.exp(model_dec.params["age_decade"]), 2))

observed = df.groupby("age_decade")["in_hospital_death"].agg(["mean", "count"])
grid = pd.DataFrame({"age_decade": np.linspace(2, 10, 100)})
fig, ax = plt.subplots()
ax.scatter(observed.index, observed["mean"] * 100, s=observed["count"] / 2, color=TEAL, alpha=0.7, label="Observed death rate")
ax.plot(grid["age_decade"], model_dec.predict(grid) * 100, color=VIOLET, lw=3, label="Logistic regression")
for dec in [5, 8]:
    r = model_dec.predict(pd.DataFrame({"age_decade": [dec]}))[0] * 100
    ax.scatter(dec, r, s=120, color=CORAL, zorder=5)
    ax.annotate(f"{r:.1f}%", (dec, r), xytext=(-45, 8), textcoords="offset points", color=CORAL, fontweight="bold")
ax.set_xlabel("Age decade"); ax.set_ylabel("Risk of death (%)"); ax.legend(loc="upper left")
ax.set_title("Each decade multiplies the odds by 1.68")
plt.show()

Several predictors: z is a linear combination of all of them. Each coefficient becomes an adjusted OR with `np.exp`. An OR above 1 raises the risk, below 1 lowers it.

In [ ]:
model_death = smf.logit("in_hospital_death ~ age_decade + heart_failure + resp_rate + spo2", data=df).fit(disp=0)
ors = np.exp(model_death.params.drop("Intercept"))
print(pd.DataFrame({"β": model_death.params, "OR": np.exp(model_death.params)}).round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
names = ["Age, per decade", "Heart failure", "Resp. rate, per breath/min", "SpO₂, per 1%"]
ax.barh(names, ors.values - 1, left=1, color=[CORAL if v > 1 else TEAL for v in ors.values])
ax.axvline(1, color=NAVY, lw=1.5)
ax.set_xlim(0.7, 1.65)
for i, v in enumerate(ors.values):
    ax.text(v + (0.02 if v > 1 else -0.02), i, f"{v:.2f}", va="center", ha="left" if v > 1 else "right", fontweight="bold")
ax.set_xlabel("Adjusted odds ratio (1 = no effect)"); ax.set_title("Heart failure: OR 2.78 crude, 1.53 adjusted")
plt.show()

### Your turn 2

Patient P00503: eighties, no heart failure, respiratory rate 25. Fill in P00503's SpO₂, 91, and run. The plot puts the patient on the logistic curve, next to the same patient breathing normally.

In [ ]:
patient = pd.DataFrame({"age_decade": [8], "heart_failure": [0], "resp_rate": [25], "spo2": [____]})
normal = pd.DataFrame({"age_decade": [8], "heart_failure": [0], "resp_rate": [16], "spo2": [97]})
risk_503, risk_normal = model_death.predict(patient)[0], model_death.predict(normal)[0]
print("Risk of in-hospital death:", round(risk_503, 3))

fig, ax = plt.subplots()
ax.plot(z_grid, 1 / (1 + np.exp(-z_grid)), color=VIOLET, lw=3)
for risk, colour, name in [(risk_normal, TEAL, "Same patient, RR 16, SpO₂ 97"), (risk_503, CORAL, "P00503 as admitted")]:
    ax.scatter(np.log(risk / (1 - risk)), risk, s=140, color=colour, zorder=5, label=f"{name}: {risk:.0%}")
ax.set_xlabel("z (log-odds)"); ax.set_ylabel("Risk of death"); ax.set_ylim(0, 1); ax.legend(loc="upper left")
plt.show()

The same model gives every admission a risk. How well these risks separate the patients who died from those who survived is the topic of Data Science 3; here is a first look.

In [ ]:
df["predicted_risk"] = model_death.predict(df)
fig, ax = plt.subplots()
bins = np.linspace(0, 0.9, 31)
for outcome, colour, name in [(0, TEAL, "Survived"), (1, CORAL, "Died")]:
    risks = df.loc[df["in_hospital_death"] == outcome, "predicted_risk"]
    ax.hist(risks, bins=bins, color=colour, alpha=0.7, weights=np.ones(len(risks)) / len(risks) * 100, label=name)
ax.axvline(model_death.predict(patient_503).iloc[0], color=NAVY, ls="--", label="P00503")
ax.set_xlabel("Predicted risk of death"); ax.set_ylabel("% of the group"); ax.legend()
ax.set_title("Patients who died mostly got higher predicted risks")
plt.show()

## Key points

- Without a model the best guess is the mean. A regression line does better: intercept = the prediction at 0, slope = the change per unit.
- Each stay = mean + what the line explains + error. R² = explained ÷ total sum of squares.
- More predictors, of any type (binary, categorical, continuous), extend the same equation. Each coefficient is the change per one unit of its predictor, with the others held fixed.
- For a yes/no outcome, logistic regression models the log-odds. With one predictor, the coefficient is log(OR).
- Risk: z = the sum of the coefficients times the patient's values; odds = e<sup>z</sup>; risk = odds ÷ (1 + odds).

## Further reading

- James, Witten, Hastie, Tibshirani and Taylor. *An Introduction to Statistical Learning with Applications in Python* (ISLP), sections 3.1–3.3 and 4.1–4.3. Free at [statlearning.com](https://www.statlearning.com).
- Enrichment notebook: scaling and significance in linear regression; the logit, odds ratios and the road to AUROC.